In [1]:
import yaml
from xlstm.xlstm_large.model import xLSTMLargeConfig, xLSTMLarge
from tokenizers import Tokenizer

%cd ../

/mnt/ssd/Code/ArtI/SL


In [4]:
import yaml
from xlstm.xlstm_large.model import xLSTMLargeConfig, xLSTMLarge
from tokenizers import Tokenizer

# ========== Config ==========

config_path = "/home/spexx/Desktop/ssd/Code/ArtI/SL/SL/configs/xlstm_large_optimal_config.yaml"
state_path = None # state_path=None will initialize a model
device = "cuda"
torch_compile_type = "default"

# ========== Init ==========

# open config
with open(config_path) as ConfigFile:
    c = yaml.safe_load(ConfigFile)

    # load tokenizer
tokenizer = Tokenizer.from_file(c["tokenizer_path"])
c["vocab_size"] = tokenizer.get_vocab_size()

# configure the model with TFLA Triton kernels
xlstm_config = xLSTMLargeConfig(
    embedding_dim=c["d_model"],
    num_heads=c["num_heads"],
    num_blocks=c["num_layers"],
    vocab_size=c["vocab_size"],
    return_last_states=c["return_last_states"],
    chunkwise_kernel=c["chunkwise_kernel"], 
    sequence_kernel=c["sequence_kernel"],
    step_kernel=c["step_kernel"],
    mode="train",
)
# instantiate the model
bot = xLSTMLarge(xlstm_config)

target_modules = []
for name, module in bot.named_modules():
    print(name)
    if(name != "embedding" and name.find("norm") == -1 and name.find("backend") == -1 and name != "lm_head" and name.count(".") == 4):
        target_modules.append(name)
print(target_modules)


embedding
backbone
backbone.blocks
backbone.blocks.0
backbone.blocks.0.norm_mlstm
backbone.blocks.0.mlstm_layer
backbone.blocks.0.mlstm_layer.q
backbone.blocks.0.mlstm_layer.k
backbone.blocks.0.mlstm_layer.v
backbone.blocks.0.mlstm_layer.ogate_preact
backbone.blocks.0.mlstm_layer.igate_preact
backbone.blocks.0.mlstm_layer.fgate_preact
backbone.blocks.0.mlstm_layer.ogate_act_fn
backbone.blocks.0.mlstm_layer.mlstm_backend
backbone.blocks.0.mlstm_layer.multihead_norm
backbone.blocks.0.mlstm_layer.out_proj
backbone.blocks.0.norm_ffn
backbone.blocks.0.ffn
backbone.blocks.0.ffn.proj_up_gate
backbone.blocks.0.ffn.proj_up
backbone.blocks.0.ffn.proj_down
backbone.blocks.0.ffn.act_fn
backbone.blocks.1
backbone.blocks.1.norm_mlstm
backbone.blocks.1.mlstm_layer
backbone.blocks.1.mlstm_layer.q
backbone.blocks.1.mlstm_layer.k
backbone.blocks.1.mlstm_layer.v
backbone.blocks.1.mlstm_layer.ogate_preact
backbone.blocks.1.mlstm_layer.igate_preact
backbone.blocks.1.mlstm_layer.fgate_preact
backbone.block

In [3]:
        # configure the model with TFLA Triton kernels
        xlstm_config = xLSTMLargeConfig(
            embedding_dim=c["d_model"],
            num_heads=c["num_heads"],
            num_blocks=c["num_layers"],
            vocab_size=c["vocab_size"],
            return_last_states=c["return_last_states"],
            chunkwise_kernel=c["chunkwise_kernel"], 
            sequence_kernel=c["sequence_kernel"],
            step_kernel=c["step_kernel"],
            mode="train",
        )
        # instantiate the model
        bot = xLSTMLarge(xlstm_config)
        bot = torch.compile(bot, mode=torch_compile_type)
        # try loading parameters
        try:
            bot.load_state_dict(torch.load(sc["model_path"], weights_only=True))
        except FileNotFoundError:
            print(f"Model {sc["model_path"]} does not exist yet, a new one will be created")

        target_modules = []
        for name, module in bot.named_modules():
            if(name != "embedding" and name.find("norm") == -1 and name.find("backend") == -1 and name != "lm_head" and name.count(".") == 4):
                target_modules.append(name)

        eva_config = EvaConfig()
        lora_config = LoraConfig( 
            r=16,
            # no embedding, pos_encoding or layernorms
            target_modules=target_modules,
            init_lora_weights="eva",
            eva_config=eva_config
        )
        peft_bot = get_peft_model(bot, lora_config)
        initialize_lora_eva_weights(peft_bot, eva_loader, prepare_model_inputs_fn = None, prepare_layer_inputs_fn = None)
        bot.train()

NameError: name 'torch' is not defined